# 01 — Scrape (smoke test)

Pulls a small sample of root-level `CLAUDE.md` files from public GitHub repos plus repo metadata, and checkpoints to Parquet.

**Set `N_TARGET` to 20 for the smoke test, then 50 for the Kaiser pilot.**

Scraping decisions (copy into the dataset card):
- Query: `filename:CLAUDE.md path:/ size:<bucket>` via the code-search REST API, restricted to root-level files.
- GitHub code search has no star/date filter, so candidates are gathered per **file-size bucket**, then repo metadata is fetched and the final sample is **stratified by repo stars**.
- Caveat: size-bucketing deliberately over-samples unusual file lengths. Fine for a pipeline test; the full run needs a sampling scheme that does not distort length statistics.
- Excluded: forks, archived repos, files over 1 MB.
- `has_tests` = a test-like directory at the repo root (`test`, `tests`, `__tests__`, `spec`, `specs`). `has_ci` = `.github/workflows`, `.travis.yml`, `.circleci`, `.gitlab-ci.yml`, `azure-pipelines.yml` or `Jenkinsfile` at the root.

In [1]:
import os, time, base64, datetime as dt
from pathlib import Path
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
TOKEN = os.environ.get("GITHUB_TOKEN")
assert TOKEN, "GITHUB_TOKEN missing from .env"

N_TARGET = 20
PER_PAGE = 10                      # candidates fetched per size bucket
SIZE_BUCKETS = ["<500", "500..2000", "2001..5000", "5001..20000", ">20000"]
STAR_BINS = [(0, 10), (10, 100), (100, 1000), (1000, 10**9)]
OUT_PATH = Path("data/raw") / f"pilot_{N_TARGET}.parquet"
SCRAPE_DATE = dt.date.today().isoformat()

session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
})
print("scrape date:", SCRAPE_DATE)

scrape date: 2026-10-19


In [2]:
def gh(url, params=None, retries=5):
    """GET with rate-limit handling. Returns the Response (404 allowed), raises otherwise."""
    for attempt in range(retries):
        r = session.get(url, params=params, timeout=30)
        if r.status_code in (403, 429) and (
            r.headers.get("x-ratelimit-remaining") == "0" or "retry-after" in r.headers
            or "secondary rate" in r.text.lower()
        ):
            if "retry-after" in r.headers:
                wait = int(r.headers["retry-after"]) + 1
            else:
                reset = int(r.headers.get("x-ratelimit-reset", time.time() + 60))
                wait = max(reset - time.time(), 0) + 2
            wait = min(wait, 120)
            print(f"rate limited, sleeping {wait:.0f}s")
            time.sleep(wait)
            continue
        if r.status_code >= 500:
            time.sleep(2 ** attempt)
            continue
        if r.status_code == 404:
            return r
        r.raise_for_status()
        return r
    raise RuntimeError(f"gave up on {url}")

## 1. Search: one query per size bucket

In [3]:
candidates, bucket_totals = [], {}
for b in SIZE_BUCKETS:
    r = gh("https://api.github.com/search/code",
           {"q": f"filename:CLAUDE.md path:/ size:{b}", "per_page": PER_PAGE})
    d = r.json()
    bucket_totals[b] = d["total_count"]
    for it in d["items"]:
        if it["path"] == "CLAUDE.md":                      # root-level only
            candidates.append({"full_name": it["repository"]["full_name"], "size_bucket": b})
    time.sleep(7)                                          # code search: 10 req/min

cand = pd.DataFrame(candidates).drop_duplicates("full_name")
print("total matches per size bucket (GitHub's count, a snapshot):")
print(pd.Series(bucket_totals))
print(f"\n{len(cand)} unique candidate repos")

total matches per size bucket (GitHub's count, a snapshot):
<500            31424
500..2000       81920
2001..5000     190976
5001..20000    236032
>20000          55936
dtype: int64

49 unique candidate repos


## 2. Repo metadata + exclusions

In [4]:
rows = []
for name in cand["full_name"]:
    r = gh(f"https://api.github.com/repos/{name}")
    if r.status_code == 404:
        continue
    m = r.json()
    rows.append({
        "full_name": name, "stars": m["stargazers_count"], "forks": m["forks_count"],
        "language": m["language"], "created_at": m["created_at"], "last_push_at": m["pushed_at"],
        "repo_size_kb": m["size"], "is_fork": m["fork"], "archived": m["archived"],
        "license": (m.get("license") or {}).get("spdx_id"),
    })
meta = cand.merge(pd.DataFrame(rows), on="full_name")
n0 = len(meta)
meta = meta[~meta.is_fork & ~meta.archived].reset_index(drop=True)
print(f"{n0} candidates -> {len(meta)} after dropping forks/archived")

49 candidates -> 49 after dropping forks/archived


## 3. Stratified sample by stars

In [5]:
def star_bin(s):
    for lo, hi in STAR_BINS:
        if lo <= s < hi:
            return f"{lo}-{hi if hi < 10**9 else 'inf'}"
meta["star_bin"] = meta["stars"].map(star_bin)

per_bin = -(-N_TARGET // len(STAR_BINS))                    # ceil
pick = meta.groupby("star_bin", group_keys=False).apply(
    lambda g: g.sample(min(len(g), per_bin), random_state=42))
if len(pick) < N_TARGET:                                     # fill from the remainder
    rest = meta.drop(pick.index)
    pick = pd.concat([pick, rest.sample(min(len(rest), N_TARGET - len(pick)), random_state=42)])
sample = pick.head(N_TARGET).reset_index(drop=True)
sample["star_bin"] = sample["stars"].map(star_bin)           # pandas 3 groupby.apply drops the key column
print(sample["star_bin"].value_counts().sort_index())

star_bin
10-100       3
100-1000    11
1000-inf     6
Name: count, dtype: int64


## 4. Fetch file content and repo-root signals

In [6]:
TEST_DIRS = {"test", "tests", "__tests__", "spec", "specs"}
CI_ROOT = {".travis.yml", ".circleci", ".gitlab-ci.yml", "azure-pipelines.yml", "jenkinsfile"}

contents, has_tests, has_ci = [], [], []
for name in sample["full_name"]:
    f = gh(f"https://api.github.com/repos/{name}/contents/CLAUDE.md")
    j = f.json() if f.status_code == 200 else {}
    if j.get("encoding") == "base64":
        contents.append(base64.b64decode(j["content"]).decode("utf-8", errors="replace"))
    else:
        contents.append(None)                                # missing or >1 MB

    root = gh(f"https://api.github.com/repos/{name}/contents/")
    names = {e["name"].lower(): e["type"] for e in root.json()} if root.status_code == 200 else {}
    has_tests.append(any(n in TEST_DIRS and t == "dir" for n, t in names.items()))
    ci = any(n in CI_ROOT for n in names)
    if ".github" in names and not ci:
        wf = gh(f"https://api.github.com/repos/{name}/contents/.github/workflows")
        ci = wf.status_code == 200
    has_ci.append(ci)

sample["content"] = contents
sample["has_tests"] = has_tests
sample["has_ci"] = has_ci
sample["n_chars"] = sample["content"].str.len()
sample["scraped_at"] = SCRAPE_DATE
sample = sample[sample["content"].notna()].reset_index(drop=True)
print(len(sample), "files with content")

20 files with content


## 5. Checkpoint to Parquet

In [7]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
sample.to_parquet(OUT_PATH, index=False)
check = pd.read_parquet(OUT_PATH)
print("saved", OUT_PATH, check.shape)
check.drop(columns="content").describe(include="all").T[["count", "unique", "top", "mean", "min", "max"]]

saved data/raw/pilot_20.parquet (20, 17)


,count,unique,top,mean,min,max
full_name,20,20,bio-ontology-research-group/mowl,NaN,NaN,NaN
size_bucket,20,5,2001..5000,NaN,NaN,NaN
stars,20.0,NaN,NaN,1412.25,86.0,10342.0
forks,20.0,NaN,NaN,183.95,6.0,1870.0
language,20,9,Python,NaN,NaN,NaN
created_at,20,20,2020-03-09T09:21:18Z,NaN,NaN,NaN
last_push_at,20,20,2026-10-01T15:02:25Z,NaN,NaN,NaN
repo_size_kb,20.0,NaN,NaN,79565.5,90.0,802033.0
is_fork,20,1,False,NaN,NaN,NaN
archived,20,1,False,NaN,NaN,NaN


In [8]:
check[["full_name", "stars", "language", "n_chars", "has_tests", "has_ci", "star_bin"]]

,full_name,stars,language,n_chars,has_tests,has_ci,star_bin
0,bio-ontology-research-group/mowl,97,Python,4394,True,True,10-100
1,sethlaw/sputr,86,Python,3191,True,False,10-100
2,SmokinCaterpillar/pypet,91,Python,4848,False,True,10-100
3,exa-labs/exa-js,131,TypeScript,2134,True,True,100-1000
4,haddocking/haddock3,280,Python,8903,True,True,100-1000
5,egraphs-good/egglog,860,Rust,1632,True,True,100-1000
6,zamesin/Next-Move-Theory-Canon-and-Skills,437,PowerShell,27582,False,False,100-1000
7,umputun/cc-thingz,487,Shell,6112,True,True,100-1000
8,hackjutsu/Lepton,10342,JavaScript,165,True,True,1000-inf
9,aquaproj/aqua,1871,Go,1112,True,True,1000-inf
